### Load packages

In [49]:
import pandas as pd
# import numpy as np
import os
# import networkx as nx
import plotly.graph_objects as go

### Define parameters and file info

In [50]:
pd.set_option('display.max_colwidth', None)

In [51]:
data_directory = r"..\data"

# -----------------------------------------------------------------------
# CSV data
# -----------------------------------------------------------------------

node_file = r"MCN_nonprofit_economy_revenue_nodes_2023.csv"
node_path = os.path.join(data_directory, node_file)
print("Node CSV file:", node_path)
if os.path.exists(node_path):
    print("EXISTS")

edge_file = r"MCN_nonprofit_economy_revenue_edges_2023.csv"
edge_path = os.path.join(data_directory, edge_file)
print("Edge CSV file:", edge_path)
if os.path.exists(edge_path):
    print("EXISTS")

# -----------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------

output_directory = r"..\output"

Node CSV file: ..\data\MCN_nonprofit_economy_revenue_nodes_2023.csv
EXISTS
Edge CSV file: ..\data\MCN_nonprofit_economy_revenue_edges_2023.csv
EXISTS


### Load data

In [52]:
# Load nonprofit economy node data

node_raw_df = pd.read_csv(node_path)
node_raw_df.head()

,Node,Node Type,X,Y
0,Donor-advised fund sponsors (national and community foundation only),Intermediary,0.55,0.36
1,Donor-advised fund sponsors (national and community foundation only) (loop),Ghost,0.57,0.34
2,Foundations (minus community foundation DAF sponsors),Intermediary,0.45,0.60
3,Foundations (minus community foundation DAF sponsors) (loop),Ghost,0.47,0.68
4,Program fees from private sources,Source,0.10,0.08


In [53]:
# Load nonprofit economy edge data

edge_raw_df = pd.read_csv(edge_path)
edge_raw_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


In [54]:
# Clean & format columns in edge data

edge_clean_df = edge_raw_df.copy()

edge_clean_df["Amount"] = pd.to_numeric(
    edge_clean_df["Amount"].replace("-", 0),
    errors="coerce"
)

edge_clean_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


### Clean up data in dataframes

In [ ]:
# Rename some of the node names
node_colname_df = node_raw_df.copy()

node_colname_df["Node Short"] = node_colname_df["Node"].replace({
    "Donor-advised fund sponsors (national and community foundation only)": "National & CF DAF sponsors",
    "Donor-advised fund sponsors (national and community foundation only) (loop)": "National & CF DAF sponsors (ghost)",
    "Foundations (minus community foundation DAF sponsors)": "Foundations",
    "Foundations (minus community foundation DAF sponsors) (loop)": "Foundations (loop)",
    "Program fees from private sources": "Program Fees",
    "State and local government": "State & local<br>government",
    "Federated Giving": "Federated giving",
    "Health (minus hospitals and nursing homes)": "Other healthcare",
    "Education (minus colleges and universities)": "Other education",
    "Public/societal benefit (minus national DAF sponsors)": "Public/societal benefit (minus national DAF sponsors)",
    "Arts, culture, humanities": "Arts & culture",
    "Environment and animals": "Environment & animals",
    "Unknown, unclassified": "Unknown"
})

node_colname_df = node_raw_df.copy()

In [ ]:
# Create a wrapped node text column
node_format_df = node_colname_df.copy()
# node_format_df["Node Wrapped"] = node_format_df["Node"].str.replace(
#     " ",
#     "<br>",
#     n=1  # only first break (keeps it readable)
# )

In [44]:
node_format_df.head()

,Node,Node Type,X,Y,Node Short,Node Wrapped
0,Donor-advised fund sponsors (national and community foundation only),Intermediary,0.55,0.36,National & CF DAF sponsors,Donor-advised<br>fund sponsors (national and community foundation only)
1,Donor-advised fund sponsors (national and community foundation only) (loop),Ghost,0.57,0.34,National & CF DAF sponsors (ghost),Donor-advised<br>fund sponsors (national and community foundation only) (loop)
2,Foundations (minus community foundation DAF sponsors),Intermediary,0.45,0.60,Foundations,Foundations<br>(minus community foundation DAF sponsors)
3,Foundations (minus community foundation DAF sponsors) (loop),Ghost,0.47,0.68,Foundations (loop),Foundations<br>(minus community foundation DAF sponsors) (loop)
4,Program fees from private sources,Source,0.10,0.08,Program Fees,Program<br>fees from private sources


In [60]:
# Prepare data for plotting

# Build a node map (convert node labels into integer indices)
node_map = {
    node: i for i, node in enumerate(node_raw_df["Node"])
}

# Convert source and recipient node into indices
sources = edge_clean_df["Source"].map(node_map)
targets = edge_clean_df["Recipient"].map(node_map)
values = edge_clean_df["Amount"]

### Visualize with Plotly Sankey (static charts)

Using Plotly Sankey because it includes the best compromises for:
- weighted edges
- self-loops (still poor)
- curved edges
- label control (still poor)
- heirarchy levels
- interactivity
- quality
- merging flows

In [61]:
# Map colors to edges and nodes
source_colors = {
    "Individuals": "rgba(31,119,180,0.35)",
    "Bequests": "rgba(255,127,14,0.35)",
    "Federated Giving": "rgba(44,160,44,0.35)",
    "Corporations": "rgba(214,39,40,0.35)",
    "State & local government": "rgba(214,39,40,0.35)",
    "Federal government": "rgba(214,39,40,0.35)",
    "Program fees": "rgba(214,39,40,0.35)",
    "Investment Income": "rgba(44,160,44,0.35)"
}
edge_colors = (
    edge_clean_df["Source"]
    .map(source_colors).fillna("rgba(160,160,160,0.35)")
)
node_colors = node_format_df["Node"].map(source_colors).fillna("rgba(200,200,200,0.4)")

# Plot the diagram
fig = go.Figure(go.Sankey(
    arrangement="snap",
    
    node=dict(
        # label=node_format_df["Node"],
        label=[""] * len(node_format_df), # remove default labels
        x=node_format_df["X"],
        y=node_format_df["Y"],
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        color=node_colors
    ),
    
    link=dict(
        # arrowlen=10,
        source=sources,
        target=targets,
        value=values,
        # color="rgba(160,160,160,0.35)"
        color=edge_colors
    )
))

# Define annotations for labels
annotations = []

source_offsets = {
    "Individuals": {"dx": 0, "dy": 0.07},   # move up
    "Bequests": {"dx": 0, "dy": -0.02}      # move down
}

for _, row in node_format_df[node_format_df["Node Type"] == "Source"].iterrows():
    node_name = row["Node"]
    dx = source_offsets.get(node_name, {}).get("dx", 0)
    dy = source_offsets.get(node_name, {}).get("dy", 0)
    annotations.append(dict(
        x=row["X"] - 0.03,
        y=1 - row["Y"],   # Plotly annotations invert y
        text=row["Node"],
        showarrow=False,
        xanchor="right",
        align="right"
    ))
for _, row in node_format_df[node_format_df["Node Type"] == "Recipient"].iterrows():
    annotations.append(dict(
        x=row["X"] + 0.03,
        y=1 - row["Y"],
        text=row["Node"],
        showarrow=False,
        xanchor="left",
        align="left"
    ))
for _, row in node_format_df[node_format_df["Node Type"] == "Intermediary"].iterrows():
    annotations.append(dict(
        x=row["X"],
        y=1 - row["Y"] - 0.01,
        text=row["Node"],
        showarrow=False,
        xanchor="center",
        align="center"
    ))

# Format
fig.update_layout(
    title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20),
    annotations=annotations
)

fig.show()

### More precise formatting

In [62]:
# Map colors to edges and nodes
source_colors = {
    "Individuals": "rgba(31,119,180,0.35)",
    "Bequests": "rgba(255,127,14,0.35)",
    "Federated Giving": "rgba(44,160,44,0.35)",
    "Corporations": "rgba(214,39,40,0.35)",
    "State and local government": "rgba(214,39,40,0.35)",
    "Federal government": "rgba(214,39,40,0.35)",
    "Program fees from private sources": "rgba(160,160,160,0.35)",
    "Investment Income": "rgba(44,160,44,0.35)"
}
edge_colors = (
    edge_clean_df["Source"]
    .map(source_colors)
    .fillna("rgba(160,160,160,0.35)")
)
node_colors = node_format_df["Node"].map(source_colors).fillna("rgba(200,200,200,0.4)")

# Plot the diagram
fig = go.Figure(go.Sankey(
    arrangement="snap",
    
    node=dict(
        # label=node_format_df["Node"],
        label=[""] * len(node_format_df), # remove default labels
        x=node_format_df["X"],
        y=node_format_df["Y"],
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        color=node_colors
    ),
    
    link=dict(
        # arrowlen=10,
        source=sources,
        target=targets,
        value=values,
        # color="rgba(160,160,160,0.35)"
        color=edge_colors
    )
))

# Define annotations for labels
annotations = []

source_offsets = {
    "Individuals": {"dx": 0, "dy": 0.07},   # move up
    "Bequests": {"dx": 0, "dy": -0.02}      # move down
}

for _, row in node_raw_df[node_raw_df["Node Type"] == "Source"].iterrows():
    node_name = row["Node"]
    dx = source_offsets.get(node_name, {}).get("dx", 0)
    dy = source_offsets.get(node_name, {}).get("dy", 0)
    annotations.append(dict(
        x=row["X"] - 0.03,
        y=1 - row["Y"],   # Plotly annotations invert y
        text=row["Node"],
        showarrow=False,
        xanchor="right",
        align="right",
        textangle=-90
    ))
for _, row in node_raw_df[node_raw_df["Node Type"] == "Recipient"].iterrows():
    annotations.append(dict(
        x=row["X"] + 0.03,
        y=1 - row["Y"],
        text=row["Node"],
        showarrow=False,
        xanchor="left",
        align="left",
        textangle=-90
    ))
for _, row in node_raw_df[node_raw_df["Node Type"] == "Intermediary"].iterrows():
    annotations.append(dict(
        x=row["X"],
        y=1 - row["Y"] - 0.01,
        text=row["Node"],
        showarrow=False,
        xanchor="center",
        align="center",
        textangle=-90
    ))

# Format
fig.update_layout(
    title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20),
    annotations=annotations
)

fig.show()

### Output image

In [11]:
fig_output_file = "MCN_nonprofit_economy_sankey_2023"
fig_output_path = os.path.join(output_directory, fig_output_file + ".png")

fig.write_image(fig_output_path)